In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
INPUT_PATH = "/content/drive/MyDrive/KOPIS/noncomplex2324_ffinal.csv"
OUT_DIR    = "/content/drive/MyDrive/KOPIS/Hyeon"

import numpy as np
import pandas as pd
import os; os.makedirs(OUT_DIR, exist_ok=True)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='matplotlib')


In [ ]:
# 저변 중심 가중치
W_U_PAID, W_S_PAID, W_A_PAID = 0.50, 0.30, 0.20
W_U_FREE, W_S_FREE           = 0.60, 0.40

# 접근성(편의) 보너스 민감도/캡
BETA_INCLUSION   = 0.10
BONUS_MULTIPLIER_CAP = (0.90, 1.10)

# 가드레일(유료, 회차당매출 하위20% → ×0.95)
GUARD_PENALTY    = 0.95

usecols = [
    "공연명","예매금액_합계","판매좌석수_합계","총_공연회차_계산","최종_좌석점유율",
    "평균_판매좌석수","평균판매단가","회차당매출","접근성_점수",
    "장르명","지역","공연시설명", "좌석수"
    # 아래 3개는 계산에는 쓰지 않지만 리포트에 포함 가능(있으면 로드됨)
]

# 인코딩 자동 처리
loaded = False
for enc in ["utf-8-sig","cp949","utf-8"]:
    try:
        df = pd.read_csv(INPUT_PATH, usecols=[c for c in usecols if c], encoding=enc)
        print(f"Loaded with {enc} →", df.shape)
        loaded = True
        break
    except FileNotFoundError:
        raise
    except Exception as e:
        print(enc, "failed:", e)
if not loaded:
    raise RuntimeError("CSV 로드 실패: 인코딩/경로 확인 필요")


Loaded with utf-8-sig → (40394, 13)


In [ ]:
# =========================
# 2) 전처리 & 유틸 (전역 기준)
# =========================

# 숫자형 변환
num_cols = ["예매금액_합계", "판매좌석수_합계", "총_공연회차_계산", "최종_좌석점유율",
            "평균_판매좌석수", "평균판매단가", "회차당매출", "접근성_점수"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# 접근성 점수 0~11로 클립
if "접근성_점수" in df.columns:
    df["접근성_점수"] = df["접근성_점수"].clip(lower=0, upper=11)

# 이상치 제거: 최종_좌석점유율 (IQR 기준)
if "최종_좌석점유율" in df.columns:
    Q1 = df["최종_좌석점유율"].quantile(0.25)
    Q3 = df["최종_좌석점유율"].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df["최종_좌석점유율"] >= lower_bound) & (df["최종_좌석점유율"] <= upper_bound)]

# 전역 윈저/백분위(코호트 없음)
def winsor_global(s: pd.Series, a=0.05, b=0.95):
    lo, hi = s.quantile(a), s.quantile(b)
    return s.clip(lo, hi)

def pct_rank_global(s: pd.Series):
    return s.rank(pct=True, method="average")


In [ ]:
# =========================
# 3) U/S/A 계산 (전역 정규화)
# =========================
# U: 참여(좌석활용)
seat_occ = df["최종_좌석점유율"].astype(float).clip(upper=1.0)
p_occ = pct_rank_global(winsor_global(seat_occ))
U = p_occ.clip(0,1)

# S: 접점/규모
p_perf = pct_rank_global(winsor_global(np.log1p(df["총_공연회차_계산"].fillna(0).clip(lower=0))))
p_sold = pct_rank_global(winsor_global(np.log1p(df["판매좌석수_합계"].fillna(0).clip(lower=0))))
p_avg  = pct_rank_global(winsor_global(df["평균_판매좌석수"].astype(float)))
S = (0.5*p_perf + 0.3*p_sold + 0.2*p_avg).clip(0,1)

# A: 가격 접근성
price = pd.to_numeric(df["평균판매단가"], errors="coerce")
rev   = pd.to_numeric(df["회차당매출"], errors="coerce")

# 백분위 (0~1)
pr = price.rank(pct=True, method="max")
rr = rev.rank(pct=True, method="max")

# 1) 회차당매출 점수
A_rev = rr

# 2) 가격 점수
A_price = 1 - pr
A_price = np.where(pr < 0.5, 0.5, A_price)

# 3) 통합 (가중 기하평균)
wr, wp = 0.6, 0.4
A = (np.clip(A_rev, 1e-9, 1)**wr) * (np.clip(A_price, 1e-9, 1)**wp)
A = A ** (1.0 / (wr + wp))
df["A"] = A


# 유/무료 구분
is_free = ((df["평균판매단가"].fillna(0) <= 0) | (df["회차당매출"].fillna(0) <= 0))

# Core DPI (기하평균)
def geo(u,s,a,wu,ws,wa):
    g = (np.clip(u,1e-9,1)**wu) * (np.clip(s,1e-9,1)**ws) * (np.clip(a,1e-9,1)**wa)
    return 100 * (g ** (1.0/(wu+ws+wa)))

DPI_core_paid = geo(U,S,A, W_U_PAID, W_S_PAID, W_A_PAID)
DPI_core_free = 100 * ((np.clip(U,1e-9,1)**W_U_FREE)*(np.clip(S,1e-9,1)**W_S_FREE))
DPI_core = np.where(is_free, DPI_core_free, DPI_core_paid)

In [ ]:
# =========================
# 4) 접근성 보너스
# =========================

I_raw  = (df["접근성_점수"].fillna(0) / 11.0).clip(0,1)
I_pct  = pct_rank_global(winsor_global(I_raw))
I_tame = np.sqrt(I_pct)
Ibar   = float(I_tame.mean())

mult   = 1 + BETA_INCLUSION * (I_tame - Ibar)
mult   = mult.clip(BONUS_MULTIPLIER_CAP[0], BONUS_MULTIPLIER_CAP[1])

DPI_plus = DPI_core * mult

# 가드레일
p_rev = pct_rank_global(winsor_global(df["회차당매출"].astype(float)))
guard = np.where((~is_free) & (p_rev < 0.20), GUARD_PENALTY, 1.00)
DPI_plus = DPI_plus * guard

In [ ]:

df = df.copy()

# 1) U/S/A/무료여부 -> 표준 컬럼 (Series로 등록, 인덱스 정렬)
df["U_participation"] = pd.to_numeric(pd.Series(U, index=df.index), errors="coerce").astype(float)
df["S_scale"]         = pd.to_numeric(pd.Series(S, index=df.index), errors="coerce").astype(float)
df["A_afford"]        = pd.to_numeric(pd.Series(A, index=df.index), errors="coerce").astype(float)
df["is_free"]         = pd.Series(is_free, index=df.index).astype(bool)


dpi_core_s = pd.to_numeric(pd.Series(DPI_core, index=df.index), errors="coerce")

if "DPI_plus" in df.columns:
    df["DPI_plus"] = pd.to_numeric(df["DPI_plus"], errors="coerce")
    df["DPI_plus"] = df["DPI_plus"].combine_first(dpi_core_s)
else:
    df["DPI_plus"] = dpi_core_s

_need = ["U_participation","S_scale","A_afford","is_free","DPI_plus"]
_missing = [c for c in _need if c not in df.columns or df[c].isna().all()]
if _missing:
    raise RuntimeError(f"필수 컬럼 누락/전부 NaN: {_missing}. U/S/A/DPI_core 계산 확인 필요.")


In [ ]:
# ==========================================
# 세그먼트별(K, CAP) 하이퍼파라미터 탐색
# ==========================================

df["is_free"] = (price <= 0) | (rev <= 0)

df["segment"] = np.where(df["is_free"], "Free", "Paid")

# ---- 필수 컬럼 체크
for c in ["DPI_plus", "지역", "segment"]:
    if c not in df.columns:
        raise KeyError(f"필수 컬럼 없음: {c}")

# ---- 평가 그리드
K_GRID   = [30, 60, 80, 100, 150]     # 수축 정도
CAP_GRID = [2.0, 3.0, 4.0, 5.0, 6.0]  # 이동 한도(점수)

# ---- 스코어 가중치(원하면 조정)
W_ETA   = 0.40   # 지역효과(η²)
W_RMS   = 0.30   # 지역 평균의 RMS 편차
W_DIST  = 0.15   # 왜곡 크기(Δ)의 95퍼센타일
W_RANK  = 0.15   # (1 - Spearman) 낮을수록 좋음 → Spearman 높을수록 좋음
RANK_FLOOR = 0.90  # 스피어만이 이보다 낮으면 패널티 부여
RANK_PENALTY = 0.20  # 패널티 가산

def _eta2(y: pd.Series, by: pd.Series) -> float:
    """ANOVA"""
    s = pd.to_numeric(y, errors="coerce")
    m = s.mean()
    # 그룹별 평균과 크기
    g = s.groupby(by, dropna=False)
    mu = g.transform("mean")
    n  = g.transform("size")
    ss_between = ((mu - m)**2).groupby(by, dropna=False).apply(lambda z: z.iloc[0] * n.loc[z.index[0]]).sum()
    ss_total   = ((s - m)**2).sum()
    if ss_total == 0 or np.isnan(ss_total):
        return 0.0
    return float(ss_between / ss_total)

def _region_rms(y: pd.Series, by: pd.Series) -> float:
    """지역 평균의 전역평균 대비 RMS 편차"""
    s = pd.to_numeric(y, errors="coerce")
    m = s.mean()
    g = s.groupby(by, dropna=False)
    mu = g.mean()
    n  = g.size()
    if n.sum() == 0:
        return 0.0
    w = n / n.sum()
    return float(np.sqrt(((mu - m)**2 * w).sum()))

def _adjust_segment(seg_df: pd.DataFrame, K: float, CAP: float) -> pd.Series:
    """세그먼트 내부에서 지역별 디밍+수축+캡 적용 후 반환."""
    s  = pd.to_numeric(seg_df["DPI_plus"], errors="coerce")
    by = seg_df["지역"]
    g  = s.groupby(by, dropna=False)
    mu_region = g.transform("mean")
    mu_all    = float(s.mean())
    n_region  = g.transform("size")
    alpha = (n_region / (n_region + K)).clip(0,1)
    delta = np.clip(alpha * (mu_region - mu_all), -CAP, CAP)
    adj = s - delta
    return adj, (adj - s), np.abs(adj - s) >= (CAP - 1e-12)

def _spearman(a: pd.Series, b: pd.Series) -> float:
    try:
        r = a.corr(b, method="spearman")
        if pd.isna(r):
            return 1.0
        return float(r)
    except Exception:
        return 1.0

def _top_overlap(a: pd.Series, b: pd.Series, n=10) -> float:
    ia = a.rank(ascending=False, method="first").nsmallest(n).index
    ib = b.rank(ascending=False, method="first").nsmallest(n).index
    return len(set(ia) & set(ib)) / max(1, n)

def evaluate_segment(seg_name: str, seg_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    y0 = pd.to_numeric(seg_df["DPI_plus"], errors="coerce")
    by = seg_df["지역"]
    eta_pre = _eta2(y0, by)
    rms_pre = _region_rms(y0, by)

    for K in K_GRID:
        for CAP in CAP_GRID:
            adj, delta, sat = _adjust_segment(seg_df, K, CAP)
            eta_post = _eta2(adj, by)
            rms_post = _region_rms(adj, by)
            sp       = _spearman(y0, adj)
            d_mean   = float(np.abs(delta).mean())
            d_p95    = float(np.percentile(np.abs(delta.dropna()), 95)) if delta.notna().any() else 0.0
            sat_rate = float(np.mean(sat)) if len(sat) else 0.0
            top_ov   = _top_overlap(y0, adj, n=10)

            rows.append({
                "segment": seg_name,
                "K": K, "CAP": CAP,
                "eta_pre": eta_pre, "eta_post": eta_post, "eta_reduction": eta_pre - eta_post,
                "rms_pre": rms_pre, "rms_post": rms_post, "rms_reduction": rms_pre - rms_post,
                "spearman": sp, "top10_overlap": top_ov,
                "dist_mean_abs": d_mean, "dist_p95": d_p95, "cap_saturation_rate": sat_rate
            })
    res = pd.DataFrame(rows)

    # min-max 정규화
    def _minmax(s):
        v = s.values.astype(float)
        v = (v - np.nanmin(v)) / (np.nanmax(v) - np.nanmin(v) + 1e-12)
        return pd.Series(v, index=s.index)

    if not res.empty:
        bad_eta  = _minmax(res["eta_post"])
        bad_rms  = _minmax(res["rms_post"])
        bad_dist = _minmax(res["dist_p95"])
        bad_rank = _minmax(1 - res["spearman"])
        score = W_ETA*bad_eta + W_RMS*bad_rms + W_DIST*bad_dist + W_RANK*bad_rank

        score += np.where(res["spearman"] < RANK_FLOOR, RANK_PENALTY, 0.0)
        res["score"] = score

        res = res.sort_values(["score","eta_post","rms_post","dist_p95"], ascending=True).reset_index(drop=True)
    return res

# 세그먼트별 평가 실행
results = []
for seg in ["Paid","Free"]:
    seg_df = df[df["segment"] == seg]
    if len(seg_df) == 0:
        print(f"{seg}: 데이터 없음")
        continue
    res = evaluate_segment(seg, seg_df)
    results.append(res)
    print(f"\n=== 후보표: {seg} (상위 10) ===")
    display(res.head(10))

# 최적값 자동 선택 및 적용(원하면 True로)
APPLY_BEST = True

if results:
    best_params = {}
    for res in results:
        seg = res.loc[0, "segment"]
        K_best  = res.loc[0, "K"]
        CAP_best= res.loc[0, "CAP"]
        best_params[seg] = {"K": float(K_best), "CAP": float(CAP_best)}
        print(f"[BEST] {seg} → K={K_best}, CAP={CAP_best} | score={res.loc[0,'score']:.4f}, "
              f"eta_post={res.loc[0,'eta_post']:.4f}, spearman={res.loc[0,'spearman']:.3f}, "
              f"rms_post={res.loc[0,'rms_post']:.3f}, d95={res.loc[0,'dist_p95']:.2f}, cap_sat={res.loc[0,'cap_saturation_rate']:.2%}")

    if APPLY_BEST:
        for c in ["DPI_region_adj_seg","DPI_region_pr_seg","region_delta_seg"]:
            df[c] = np.nan

        for seg, hp in best_params.items():
            idx = (df["segment"] == seg)
            if not idx.any(): continue
            seg_df = df.loc[idx]

            adj, delta, _ = _adjust_segment(seg_df, hp["K"], hp["CAP"])
            pr = seg_df["DPI_plus"].groupby(seg_df["지역"], dropna=False).rank(pct=True, method="average")
            df.loc[idx, "DPI_region_adj_seg"] = adj.values
            df.loc[idx, "DPI_region_pr_seg"]  = pr.values
            df.loc[idx, "region_delta_seg"]   = (adj - seg_df["DPI_plus"]).values

        print("\n✅ 최적 K/CAP 적용 완료 ")
else:
    print("평가 결과가 없습니다.")



=== 후보표: Paid (상위 10) ===


,segment,K,CAP,eta_pre,eta_post,eta_reduction,rms_pre,rms_post,rms_reduction,spearman,top10_overlap,dist_mean_abs,dist_p95,cap_saturation_rate,score
0,Paid,30,4.0,0.026678,0.000604,0.026075,2.907968,0.431632,2.476336,0.988852,0.1,2.343669,4.000000,0.244257,0.281406
1,Paid,60,4.0,0.026678,0.000629,0.026049,2.907968,0.440546,2.467422,0.988953,0.1,2.333255,4.000000,0.220232,0.283419
2,Paid,80,4.0,0.026678,0.000641,0.026037,2.907968,0.444739,2.463229,0.989025,0.1,2.325991,4.000000,0.220232,0.283959
3,Paid,100,4.0,0.026678,0.000652,0.026026,2.907968,0.448700,2.459268,0.989095,0.1,2.318948,4.000000,0.220232,0.284451
4,Paid,150,4.0,0.026678,0.000682,0.025996,2.907968,0.458889,2.449079,0.989257,0.1,2.302167,4.000000,0.220232,0.286022
5,Paid,30,5.0,0.026678,0.000220,0.026458,2.907968,0.260794,2.647174,0.987326,0.1,2.454487,5.000000,0.059579,0.287686
6,Paid,60,5.0,0.026678,0.000248,0.026430,2.907968,0.276889,2.631079,0.987493,0.1,2.439341,5.000000,0.059579,0.290535
7,Paid,80,5.0,0.026678,0.000262,0.026416,2.907968,0.284467,2.623501,0.987600,0.1,2.429603,5.000000,0.059579,0.291440
8,Paid,100,5.0,0.026678,0.000276,0.026402,2.907968,0.291725,2.616243,0.987703,0.1,2.420103,5.000000,0.059579,0.292297
9,Paid,150,5.0,0.026678,0.000315,0.026363,2.907968,0.311868,2.596100,0.987969,0.1,2.395876,4.976832,0.000000,0.294096



=== 후보표: Free (상위 10) ===


,segment,K,CAP,eta_pre,eta_post,eta_reduction,rms_pre,rms_post,rms_reduction,spearman,top10_overlap,dist_mean_abs,dist_p95,cap_saturation_rate,score
0,Free,30,6.0,0.080561,0.004210,0.076350,6.199707,1.361923,4.837783,0.968189,0.3,4.657130,6.0,0.538197,0.300000
1,Free,60,6.0,0.080561,0.004499,0.076062,6.199707,1.408031,4.791676,0.968997,0.4,4.564544,6.0,0.538197,0.302691
2,Free,80,6.0,0.080561,0.004711,0.075850,6.199707,1.440939,4.758768,0.969323,0.4,4.512051,6.0,0.538197,0.306189
3,Free,100,6.0,0.080561,0.004927,0.075634,6.199707,1.473757,4.725950,0.969621,0.4,4.465102,6.0,0.538197,0.309889
4,Free,150,6.0,0.080561,0.005460,0.075101,6.199707,1.551913,4.647794,0.970152,0.4,4.365814,6.0,0.538197,0.319969
5,Free,30,5.0,0.080561,0.009670,0.070891,6.199707,2.069660,4.130047,0.974871,0.4,4.118933,5.0,0.538197,0.349118
6,Free,60,5.0,0.080561,0.009956,0.070605,6.199707,2.100300,4.099407,0.975531,0.4,4.026347,5.0,0.538197,0.351150
7,Free,80,5.0,0.080561,0.010165,0.070396,6.199707,2.122507,4.077200,0.975877,0.4,3.973854,5.0,0.538197,0.353443
8,Free,100,5.0,0.080561,0.010379,0.070182,6.199707,2.144926,4.054780,0.976199,0.4,3.926905,5.0,0.538197,0.355949
9,Free,150,5.0,0.080561,0.010907,0.069654,6.199707,2.199369,4.000338,0.976735,0.4,3.827617,5.0,0.538197,0.363610


[BEST] Paid → K=30, CAP=4.0 | score=0.2814, eta_post=0.0006, spearman=0.989, rms_post=0.432, d95=4.00, cap_sat=24.43%
[BEST] Free → K=30, CAP=6.0 | score=0.3000, eta_post=0.0042, spearman=0.968, rms_post=1.362, d95=6.00, cap_sat=53.82%

✅ 최적 K/CAP 적용 완료 → `DPI_region_adj_seg` 사용하세요.


In [ ]:
# =========================
# 6) 저장(정렬/파일 분리)
# =========================

# 보고서 컬럼(있는 것만 사용)
desired_cols = [
    "공연명","장르명","지역","공연시설명",
    "최종_좌석점유율","총_공연회차_계산","판매좌석수_합계","평균_판매좌석수",
    "평균판매단가","회차당매출","접근성_점수", "좌석수",
    "U_participation","S_scale","A_afford","DPI_core","DPI_plus",
    "DPI_region_adj_seg","DPI_region_pr_seg","is_free"
]
rank_cols = [c for c in desired_cols if c in df.columns]

# 정렬 키
sort_keys = ["DPI_region_adj_seg","U_participation","S_scale","A_afford"]

# 세그먼트별 저장 (Paid / Free)
for seg in ["Paid", "Free"]:
    seg_df = df[df["segment"] == seg].copy()
    if seg_df.empty:
        print(f"{seg}: 저장할 데이터가 없습니다.")
        continue

    seg_rank = seg_df.sort_values(sort_keys, ascending=[False,False,False,False])[rank_cols]


    out_seg = f"{OUT_DIR}/DPI_ranked_{seg}_noncomplex.csv"
    seg_rank.to_csv(out_seg, index=False, encoding="utf-8-sig")
    print("Saved:", out_seg, "| rows:", len(seg_rank))

Saved: /content/drive/MyDrive/KOPIS/Hyeon/DPI_ranked_Paid_noncomplex.csv | rows: 34173
Saved: /content/drive/MyDrive/KOPIS/Hyeon/DPI_ranked_Free_noncomplex.csv | rows: 4071
